<a href="https://colab.research.google.com/github/HuyLuong2002/anfis-breast-cancer/blob/main/anfis_breast_cancer_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài đặt thư viện



In [18]:
# Chạy cell này trước tiên
!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn ucimlrepo

!git clone https://github.com/hudscomdz/scikit-anfis.git
%cd scikit-anfis
!pip install -q -r requirements.txt
!pip install -q .
%cd ..

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import IterativeImputer
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import skanfis as ANFIS
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # Đảm bảo kết quả tái lập được

fatal: destination path 'scikit-anfis' already exists and is not an empty directory.
/content/scikit-anfis
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/content


# Nạp dữ liệu & Tiền xử lý (MICE + Loại ngoại lai)

In [19]:
# 1. Tải dữ liệu gốc
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
cols = ['ID', 'Clump_Thickness', 'Uniformity_Cell_Size', 'Uniformity_Cell_Shape',
        'Marginal_Adhesion', 'Single_Epithelial_Cell_Size', 'Bare_Nuclei',
        'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses', 'Class']

df_raw = pd.read_csv(url, names=cols, na_values='?')
print(f"✅ Dữ liệu gốc: {df_raw.shape[0]} mẫu × {df_raw.shape[1]} thuộc tính")
display(df_raw.head(10).style.set_caption("📋 10 mẫu đầu tiên của dữ liệu gốc"))

# Kiểm tra missing values
missing = df_raw.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠️ Phát hiện {missing.sum()} giá trị thiếu:")
    display(missing[missing > 0].to_frame(name='Số lượng thiếu').style.set_caption("🔍 Các cột có missing values"))

# 2. Mã hóa nhãn và tách features
print("\n" + "="*70)
print("📊 BƯỚC 2: MÃ HÓA NHÃN & TÁCH FEATURES")
print("="*70)

df = df_raw.copy()
df.drop('ID', axis=1, inplace=True)
df['Class'] = df['Class'].map({2: 0, 4: 1})  # 0: Benign, 1: Malignant

feature_names = df.columns[:-1].tolist()
X = df[feature_names].values
y = df['Class'].values

print(f"✅ Features: {feature_names}")
print(f"✅ Label distribution: Benign={sum(y==0)}, Malignant={sum(y==1)}")
display(pd.DataFrame({'Class': ['Benign', 'Malignant'],
                      'Count': [sum(y==0), sum(y==1)]}).style.set_caption("📈 Phân bố nhãn"))

# 3. Xử lý missing values bằng MICE
print("\n" + "="*70)
print("📊 BƯỚC 3: XỬ LÝ MISSING VALUES BẰNG MICE (IterativeImputer)")
print("="*70)

print(f"🔹 Trước MICE: Số missing = {pd.DataFrame(X, columns=feature_names).isnull().sum().sum()}")
imputer = IterativeImputer(max_iter=10, random_state=42, verbose=0)
X_imputed = imputer.fit_transform(X)
print(f"✅ Sau MICE: Số missing = {pd.DataFrame(X_imputed, columns=feature_names).isnull().sum().sum()}")

# Hiển thị so sánh 5 mẫu có missing ban đầu
missing_idx = df_raw['Bare_Nuclei'].isnull()
if missing_idx.any():
    sample_idx = df_raw[missing_idx].head(5).index
    print(f"\n🔍 Ví dụ: 5 mẫu có 'Bare_Nuclei' bị thiếu → đã được điền khuyết:")
    comparison = pd.DataFrame({
        'Bare_Nuclei_Original': df_raw.loc[sample_idx, 'Bare_Nuclei'],
        'Bare_Nuclei_Imputed': np.round(X_imputed[sample_idx, feature_names.index('Bare_Nuclei')], 2)
    })
    display(comparison.style.set_caption("🔄 So sánh trước/sau MICE"))

# 4. Phát hiện & loại bỏ outliers bằng Euclidean Distance
print("\n" + "="*70)
print("📊 BƯỚC 4: PHÁT HIỆN & LOẠI BỎ OUTLIERS (Euclidean Distance, ngưỡng 95%)")
print("="*70)

mean_vec = np.mean(X_imputed, axis=0)
dists = np.sqrt(np.sum((X_imputed - mean_vec)**2, axis=1))
threshold = np.percentile(dists, 95)
outlier_mask = dists > threshold

print(f"🔹 Tổng mẫu: {len(dists)}")
print(f"🔹 Ngưỡng Euclidean (95th percentile): {threshold:.3f}")
print(f"⚠️ Số outliers phát hiện: {outlier_mask.sum()} ({outlier_mask.mean()*100:.1f}%)")

# Hiển thị thông tin outliers
if outlier_mask.any():
    outlier_stats = pd.DataFrame({
        'Feature': feature_names,
        'Mean_All': np.mean(X_imputed, axis=0),
        'Mean_Outliers': np.mean(X_imputed[outlier_mask], axis=0),
        'Std_All': np.std(X_imputed, axis=0)
    }).round(2)
    display(outlier_stats.style.set_caption("📊 Thống kê: Outliers vs Toàn bộ dữ liệu"))

# Loại outliers
X_clean = X_imputed[~outlier_mask]
y_clean = y[~outlier_mask]
print(f"✅ Sau khi loại outliers: {X_clean.shape[0]} mẫu còn lại")

# 5. Novel Relief Algorithm (Chọn 4 đặc trưng)
print("\n" + "="*70)
print("📊 BƯỚC 5: LỰA CHỌN ĐẶC TRƯNG VỚI NOVEL RELIEF (Mahalanobis + τ=0.5)")
print("="*70)

def novel_relief(X, y, m=475, tau=0.5):
    """Novel Relief với Mahalanobis distance, xử lý missing/outliers đã được tiền xử lý"""
    n, a = X.shape
    W = np.zeros(a)
    cov = np.cov(X.T) + 1e-5 * np.eye(a)  # Thêm epsilon để ổn định
    inv_cov = np.linalg.inv(cov)
    rng = np.ptp(X, axis=0); rng[rng==0] = 1  # Chuẩn hóa theo range

    for _ in range(m):
        i = np.random.randint(n)
        Ri, yi = X[i], y[i]
        same_mask = (y == yi)
        diff_mask = (y != yi)

        # Tìm Nearest Hit & Nearest Miss bằng Mahalanobis
        d_hit = np.array([mahalanobis(Ri, X[j], inv_cov) for j in np.where(same_mask)[0]])
        d_miss = np.array([mahalanobis(Ri, X[j], inv_cov) for j in np.where(diff_mask)[0]])

        Hi = X[same_mask][np.argmin(d_hit)]
        Mi = X[diff_mask][np.argmin(d_miss)]

        # Cập nhật trọng số đặc trưng
        for j in range(a):
            diff_h = abs(Ri[j] - Hi[j]) / rng[j]
            diff_m = abs(Ri[j] - Mi[j]) / rng[j]
            W[j] += (diff_m - diff_h)

    W /= m  # Trung bình hóa
    return W, np.where(W >= tau)[0]

# Chạy Novel Relief
weights, selected_idx = novel_relief(X_clean, y_clean, m=475, tau=0.5)

# Hiển thị kết quả chọn đặc trưng
feature_scores = pd.DataFrame({
    'Feature': feature_names,
    'Relevance_Score': np.round(weights, 4),
    'Selected': ['✅ YES' if i in selected_idx else '❌ NO' for i in range(len(feature_names))]
}).sort_values('Relevance_Score', ascending=False)

print(f"🎯 Ngưỡng τ = 0.5 | Số mẫu lặp m = 475")
print(f"✅ Đặc trưng được chọn ({len(selected_idx)}/{len(feature_names)}):")
for idx in selected_idx:
    print(f"   • {feature_names[idx]} (Score = {weights[idx]:.4f})")

display(feature_scores.style.set_caption("🏆 Điểm liên quan của từng đặc trưng").format({'Relevance_Score': '{:.4f}'}))

# 6. Dữ liệu cuối cùng sau tiền xử lý
print("\n" + "="*70)
print("📊 KẾT QUẢ CUỐI CÙNG SAU TIỀN XỬ LÝ")
print("="*70)

X_final = X_clean[:, selected_idx]
final_features = [feature_names[i] for i in selected_idx]

summary = pd.DataFrame({
    'Giai đoạn': ['Dữ liệu gốc', 'Sau MICE', 'Sau loại outliers', 'Sau chọn features'],
    'Số mẫu': [df_raw.shape[0], df_raw.shape[0], X_clean.shape[0], X_final.shape[0]],
    'Số features': [len(feature_names), len(feature_names), len(feature_names), len(final_features)],
    'Missing values': [df_raw.isnull().sum().sum(), 0, 0, 0],
    'Outliers': ['Chưa xử lý', 'Chưa xử lý', f'Đã loại {outlier_mask.sum()}', 'Đã loại'],
    'Features': [', '.join(feature_names), '-', '-', ', '.join(final_features)]
})

display(summary.style.set_caption("📋 Tóm tắt quá trình tiền xử lý").set_properties(**{'text-align': 'left'}))

# Hiển thị 5 mẫu đầu của dữ liệu cuối cùng
df_final = pd.DataFrame(X_final, columns=final_features)
df_final['Label'] = y_clean
print(f"\n✅ Dữ liệu sẵn sàng cho huấn luyện: {X_final.shape[0]} mẫu × {X_final.shape[1]} features")
display(df_final.head(10).style.set_caption("🎯 10 mẫu đầu của dữ liệu cuối cùng (đã chuẩn bị cho ANFIS)"))

# Lưu lại để dùng cho các cell sau
print("\n💾 Dữ liệu đã sẵn sàng! Biến xuất ra:")
print(f"   • X_final: numpy array shape {X_final.shape}")
print(f"   • y_clean: numpy array shape {y_clean.shape}")
print(f"   • final_features: list features {final_features}")

✅ Dữ liệu gốc: 699 mẫu × 11 thuộc tính


,ID,Clump_Thickness,Uniformity_Cell_Size,Uniformity_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.000000,3,1,1,2
1,1002945,5,4,4,5,7,10.000000,3,2,1,2
2,1015425,3,1,1,1,2,2.000000,3,1,1,2
3,1016277,6,8,8,1,3,4.000000,3,7,1,2
4,1017023,4,1,1,3,2,1.000000,3,1,1,2
5,1017122,8,10,10,8,7,10.000000,9,7,1,4
6,1018099,1,1,1,1,2,10.000000,3,1,1,2
7,1018561,2,1,2,1,2,1.000000,3,1,1,2
8,1033078,2,1,1,1,2,1.000000,1,1,5,2
9,1033078,4,2,1,1,2,1.000000,2,1,1,2



⚠️ Phát hiện 16 giá trị thiếu:


,Số lượng thiếu
Bare_Nuclei,16



📊 BƯỚC 2: MÃ HÓA NHÃN & TÁCH FEATURES
✅ Features: ['Clump_Thickness', 'Uniformity_Cell_Size', 'Uniformity_Cell_Shape', 'Marginal_Adhesion', 'Single_Epithelial_Cell_Size', 'Bare_Nuclei', 'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses']
✅ Label distribution: Benign=458, Malignant=241


...


📊 KẾT QUẢ CUỐI CÙNG SAU TIỀN XỬ LÝ


,Giai đoạn,Số mẫu,Số features,Missing values,Outliers,Features
0,Dữ liệu gốc,699,9,16,Chưa xử lý,"Clump_Thickness, Uniformity_Cell_Size, Uniformity_Cell_Shape, Marginal_Adhesion, Single_Epithelial_Cell_Size, Bare_Nuclei, Bland_Chromatin, Normal_Nucleoli, Mitoses"
1,Sau MICE,699,9,0,Chưa xử lý,-
2,Sau loại outliers,664,9,0,Đã loại 35,-
3,Sau chọn features,664,0,0,Đã loại,



✅ Dữ liệu sẵn sàng cho huấn luyện: 664 mẫu × 0 features


,Label
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0



💾 Dữ liệu đã sẵn sàng! Biến xuất ra:
   • X_final: numpy array shape (664, 0)
   • y_clean: numpy array shape (664,)
   • final_features: list features []


# Khởi tạo mô hình skanfis và Huấn luyện

In [20]:
import inspect
import skanfis

print("🔍 Đang khám phá cấu trúc module 'skanfis'...")
classes = {name: obj for name, obj in inspect.getmembers(skanfis) if inspect.isclass(obj)}
print(f"📦 Các class tìm thấy: {list(classes.keys())}")

# 1. Lọc bỏ các class Membership Function, chỉ giữ lại class mô hình
model_candidates = [name for name in classes.keys() if 'MembFunc' not in name]
if not model_candidates:
    raise RuntimeError("❌ Không tìm thấy class mô hình ANFIS trong skanfis!")

# 2. Chọn class mô hình (ưu tiên tên chứa 'anfis')
target_name = next((n for n in model_candidates if 'anfis' in n.lower()), model_candidates[0])
ModelClass = classes[target_name]
print(f"✅ Đang sử dụng class mô hình: {ModelClass.__name__}")

# 3. Kiểm tra signature để ánh xạ tham số chính xác
sig = inspect.signature(ModelClass.__init__)
print(f"📋 Signature __init__: {sig}")
valid_params = set(sig.parameters.keys()) - {'self'}

# Ánh xạ tham số chuẩn bài báo → tham số thực tế của thư viện
paper_params = {
    'n_memberships': 3, 'n_mf': 3,
    'membership_type': 'gaussian', 'mf_type': 'gaussian',
    'learning_rate': 0.01, 'lr': 0.01,
    'epochs': 50, 'n_epochs': 50,
    'verbose': True
}
safe_params = {k: v for k, v in paper_params.items() if k in valid_params}
print(f"🔧 Tham số hợp lệ được chọn: {safe_params}")

# 4. Khởi tạo mô hình
model = ModelClass(**safe_params)
print("🛠️ Khởi tạo mô hình thành công!")

# 5. Huấn luyện
print("📡 Đang huấn luyện ANFIS (Forward Pass: Least-Squares → Backward Pass: Gradient Descent)...")
model.fit(X_train_sc, y_train)

🔍 Đang khám phá cấu trúc module 'skanfis'...
📦 Các class tìm thấy: ['BellMembFunc', 'GaussMembFunc', 'TrapezoidalMembFunc', 'TriangularMembFunc', 'scikit_anfis']
✅ Đang sử dụng class mô hình: scikit_anfis
📋 Signature __init__: (self, fs=None, data=None, outvarnames=['y0'], epoch=10, description=None, rules=None, label='r', hybrid=True, zerotype=False, show_banner=False)
🔧 Tham số hợp lệ được chọn: {}


AttributeError: 'NoneType' object has no attribute 'items'

# Đánh giá và Visualization

In [ ]:
# Dự đoán & Đánh giá
y_pred = model.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)

print(f"\n📈 Kết quả tập Test:")
print(f"✅ Accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout(); plt.show()

# Training History (nếu thư viện lưu)
if hasattr(model, 'history_') or hasattr(model, 'errors_'):
    hist = getattr(model, 'history_', getattr(model, 'errors_', None))
    if hist is not None:
        plt.plot(hist, marker='o', color='teal')
        plt.xlabel('Epoch'); plt.ylabel('MSE')
        plt.title('ANFIS Training Convergence')
        plt.grid(True, alpha=0.3); plt.show()

# Convert HTML

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab_Notebooks/anfis-breast-cancer-model-training" --output "030626_nhat_ky_lan_1"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab_Notebooks/anfis-breast-cancer-model-training to html
[NbConvertApp] Writing 320275 bytes to /content/drive/MyDrive/Colab_Notebooks/030626_nhat_ky_lan_1.html
